In [1]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()


Successfully saved authorization token.


In [2]:
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

### Method #1 for buffering the feature collection (did not work)

In [3]:
feature = ee.FeatureCollection("projects/deforestation-495419/assets/panama_protected_areas_polygons").filterBounds(panama_geom)

# Cast the resulting object as an ee.Feature so that the call to the buffer
# method is unambiguous (first() and buffer() are shared by multiple classes).
feature = ee.Feature(feature)

# Generate buffered features out and in from the original boundary.
buffer_out = feature.buffer(20000)  # 20 km out

### Method #2 for buffering the feature collection

In [4]:
features = ee.FeatureCollection("projects/deforestation-495419/assets/panama_protected_areas_polygons").filterBounds(panama_geom)

# Define the buffer function (Distance is in meters: 20km = 20000m) 
def add_buffer(feature):
    return feature.buffer(20000)

# Map the function over the FeatureCollection
buffered_features = features.map(add_buffer)

In [ ]:
# Method 1: Attribute Filtering (ISO3 Code) - looks at legally designated protected areas in Panama (therefore this one is better for our purposes)
panama_pas_iso = ee.FeatureCollection("WCMC/WDPA/current/polygons").filter(
    ee.Filter.eq("ISO3", "PAN")
)

count_iso = panama_pas_iso.size().getInfo()
print(f"Count (via ISO3 code): {count_iso}")


# Method 2: Spatial Filtering (Using GAUL 2015 Boundary) - looks at whatever falls within the Panama boundary, regardless of legal designation
panama_boundary = ee.FeatureCollection("FAO/GAUL/2015/level0").filter(
    ee.Filter.eq("ADM0_NAME", "Panama")
)

panama_pas_spatial = ee.FeatureCollection("WCMC/WDPA/current/polygons").filterBounds(
    panama_boundary
)

count_spatial = panama_pas_spatial.size().getInfo()
print(f"Count (via Spatial Intersection): {count_spatial}")

Count (via ISO3 code): 78
Count (via Spatial Intersection): 87


### Quick List of Names (Fastest)

In [19]:
import ee

ee.Initialize()

# Load Panama protected areas
panama_pas = ee.FeatureCollection("WCMC/WDPA/current/polygons").filter(
    ee.Filter.eq("ISO3", "PAN")
)

# Extract English names as a Python list
english_names = panama_pas.aggregate_array("NAME").getInfo()

# Print first 10 names
print(english_names)

['Serranía de Darién', 'Narganá', 'Alto Darién', 'Serranía del Bagre', 'Darién', 'Sistemas de Humedales de Matusagaratí', 'Chepigana', 'Isla del Rey', 'Majé', 'Filo del Tallo-Canglón', 'Darién', 'Punta Patiño', 'Tapagra', 'Parc national du Darien', 'Isla Bastimentos', 'Portobelo', 'Reverendo Padre Jesús Héctor Gallego Herrera', 'Cerro Hoya', 'Sarigua', 'Coiba', 'Humedal de Bahía de Panamá', 'Isla Boná', 'Isla de Cañas', 'Taboga-Urabá', 'Isla Iguana', 'Peñón de La Honda', 'La Playa de la Barqueta Agrícola', 'Manglares de Panamá Viejo', 'Zona de Reserva Matumbal', 'Playa Bluff', 'San San Pond Sak', 'Humedal de Bahía de Panamá', 'Golfo de Montijo', 'Parc national de Coiba et sa zone spéciale de protection marine', 'Golfo de Montijo', 'Escudo de Veraguas', 'Golfo de Chiriquí', 'Playa Boca Vieja', 'Isla Montuosa', 'Pablo Arturo Barrios', 'Zona de Reserva La Marinera', 'Cordillera de Coiba', 'Banco Volcán', 'Palo Seco', 'San Lorenzo', 'Laguna de Volcán', 'Los Pozos de Calobre', 'Cerro Gaital

### Extract Names with Metadata (Pandas DataFrame)

In [20]:
import pandas as pd
import ee

ee.Initialize()

# Filter WDPA polygons for Panama
panama_pas = ee.FeatureCollection("WCMC/WDPA/current/polygons").filter(
    ee.Filter.eq("ISO3", "PAN")
)

# Select only relevant columns to minimize data transfer size
attributes = ["NAME", "ORIG_NAME", "DESIG", "IUCN_CAT", "STATUS", "REP_AREA"]

# Reduce features to a list of property dictionaries
features_list = panama_pas.reduceColumns(
    reducer=ee.Reducer.toList().repeat(len(attributes)),
    selectors=attributes
).get("list").getInfo()

# Convert to Pandas DataFrame
df = pd.DataFrame(dict(zip(attributes, features_list)))

# Sort alphabetically by original name
df = df.sort_values(by="ORIG_NAME").reset_index(drop=True)

# Display first few entries
print(df.head())

# Export to CSV
# df.to_csv("panama_protected_areas.csv", index=False)

ValueError: All arrays must be of the same length

### Extract WDPA IDs alongside Names and Polygons into a DataFrame
#### To match site IDs and polygon IDs directly with their names

In [22]:
import pandas as pd
import ee

ee.Initialize()

# Filter WDPA polygons for Panama
panama_pas = ee.FeatureCollection("WCMC/WDPA/current/polygons").filter(
    ee.Filter.eq("ISO3", "PAN")
)

# Function to extract attributes cleanly per feature
def extract_props(feature):
    props = feature.toDictionary()
    return ee.Feature(None, {
        'WDPAID': props.get('WDPAID', 'N/A'),
        'WDPA_PID': props.get('WDPA_PID', 'N/A'),
        'ORIG_NAME': props.get('ORIG_NAME', 'N/A'),
        'NAME': props.get('NAME', 'N/A'),
        'DESIG_ENG': props.get('DESIG_ENG', 'N/A')
    })

# Extract property dictionaries
dict_list = panama_pas.map(extract_props).reduceColumns(
    ee.Reducer.toList(), ['system:index'] # dummy reducer to fetch feature list
).get('list') # Returns a list of dictionaries

# Convert directly to DataFrame
features_info = panama_pas.map(extract_props).getInfo()['features']
rows = [f['properties'] for f in features_info]

df = pd.DataFrame(rows)

# Reorder columns
df = df[['WDPAID', 'WDPA_PID', 'ORIG_NAME', 'NAME', 'DESIG_ENG']]

# print(df.head(10))
print(df)

   WDPAID WDPA_PID ORIG_NAME                                  NAME  \
0     N/A      N/A       N/A                    Serranía de Darién   
1     N/A      N/A       N/A                               Narganá   
2     N/A      N/A       N/A                           Alto Darién   
3     N/A      N/A       N/A                    Serranía del Bagre   
4     N/A      N/A       N/A                                Darién   
..    ...      ...       ...                                   ...   
73    N/A      N/A       N/A        Ribera Oeste del Lago Alajuela   
74    N/A      N/A       N/A                Ciénaga de Las Macanas   
75    N/A      N/A       N/A                        Bahía de Chame   
76    N/A      N/A       N/A                                Donoso   
77    N/A      N/A       N/A  Reserva de la Biósfera de La Amistad   

                       DESIG_ENG  
0           Hydrological Reserve  
1                      Wild Area  
2              Protective Forest  
3            Biolog

In [5]:
Map.addLayer(features, {}, 'protected areas')
Map.addLayer(buffered_features, {}, '20km buffer')

Map

Map(center=[8.5158389458998, -80.10966640141521], controls=(WidgetControl(options=['position', 'transparent_bg…